In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pybromo as pbm
from pybromo.diffusion import Particle   # classe non esposta in pybromo.__init__
import os, time, tables
# directory to save HDF5
folder = './Pybromo_simulations'
os.makedirs(folder, exist_ok=True)

## System setup: **BOX** & **LASER**

In [ ]:
# grant reproducibility by defining initial random state
rs = np.random.RandomState(seed=1)
# simulate box system
box = pbm.Box(
    x1 = -2.5e-6, x2 = +2.5e-6,
    y1 = -2.5e-6, y2 = +2.5e-6,
    z1 = -5.0e-6, z2 = +5.0e-6
)
# PSF for laser source centering the excitation confocal volume
w0 = 0.25e-6 
z0 = 1.0e-6
# conversion radius -> sigma for Gaussian
sx = w0 / np.sqrt(2)
sy = w0 / np.sqrt(2)
sz = z0 / np.sqrt(2)

# ellissoid confocal volume
vol_psf = (4/3) * np.pi * (w0**2) * z0
# excited volume
vol_box = (box.x2 - box.x1) * (box.y2 - box.y1) * (box.z2 - box.z1)

# creation of laser PSF
psf = pbm.GaussianPSF(
    xc=0, yc=0, zc=0,  # Box centering
    sx=sx, sy=sy, sz=sz
)

# I. **Free EtBr** run

In [ ]:
# diffusion specs
EtBr_Du = 425.0          # µm²/s
EtBr_D = EtBr_Du*(1e-6)**2   # m²/s
# average number of molecules in confocal volume
n_avg = 1
# absolute number of particles to generate in the box 
n_particles = int(n_avg * (vol_box / vol_psf))
# time simulation parameters
t_step = 5e-6   # time step (s), t
t_max = 60       # total simulation (s)

print(f"To have {n_avg} molecules in the excitation volume,")
print(f"There will be needed {n_particles} particles in the box.")

In [ ]:
# generate particles in the simulation
P = pbm.Particles(
    num_particles = n_particles, D = EtBr_D, box = box, rs = rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, t_max=t_max, particles=P, box=box, psf=psf
)
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
# once simulation is done, it is better to close the HDF5 file to prevent issues in the workflow
# (the file will be opened again in the notebook 2 to generate photon timestamps)
S.store.close()

# II. **Bound EtBr-DNA** run (0.12 DNA 'blobs' in confocal vol.)

In [ ]:
# Population 1 (free EtBr) specs
# average number of molecules in confocal volume
n_avg_free = 1.0
N_EtBr_free = int(n_avg_free * (vol_box / vol_psf))
D_free = 425.0 * (1e-6)**2   # m²/s

# Population 2 (bounded EtBr-DNA) specs
# average number of 'blobs' of DNA (with bounded EtBr) in confocal volume
n_avg_bound = 0.12
N_blobs = max(1, int(n_avg_bound * (vol_box / vol_psf)))
D_bound = 1.0 * (1e-6)**2
# time simulation parameters
t_step = 5e-6   # time step (s), t
t_max = 60       # total simulation (s)

print(f'EtBr-free specs:\n- Diffusion: {D_free} m^2/s\n- Avg in confocal volume: {n_avg_free}\n- Absolute number: {N_EtBr_free}')
print(f'EtBr-DNA  specs:\n- Diffusion: {D_bound} m^2/s\n- Avg in confocal volume: {n_avg_bound}\n- Absolute number: {N_blobs}')

In [ ]:
# generate particles in the simulation
P = pbm.Particles.from_specs(
    num_particles=(N_EtBr_free, N_blobs),
    D=(D_free, D_bound),
    box=box, rs=rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, 
    t_max=t_max, 
    particles=P, 
    box=box, 
    psf=psf
    )
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
# once simulation is done, it is better to close the HDF5 file to prevent issues in the workflow
# (the file will be opened again in the notebook 2 to generate photon timestamps)
S.store.close()

# II. **Bound EtBr-DNA** run (0.18 DNA 'blobs' in confocal vol.)

In [ ]:
# Population 1 (free EtBr) specs
# average number of molecules in confocal volume
n_avg_free = 1.0
N_EtBr_free = int(n_avg_free * (vol_box / vol_psf))
D_free = 425.0 * (1e-6)**2   # m²/s

# Population 2 (bounded EtBr-DNA) specs
# average number of 'blobs' of DNA (with bounded EtBr) in confocal volume
n_avg_bound = 0.18
N_blobs = max(1, int(n_avg_bound * (vol_box / vol_psf)))
D_bound = 1.0 * (1e-6)**2
# time simulation parameters
t_step = 5e-6   # time step (s), t
t_max = 60       # total simulation (s)

print(f'EtBr-free specs:\n- Diffusion: {D_free} m^2/s\n- Avg in confocal volume: {n_avg_free}\n- Absolute number: {N_EtBr_free}')
print(f'EtBr-DNA  specs:\n- Diffusion: {D_bound} m^2/s\n- Avg in confocal volume: {n_avg_bound}\n- Absolute number: {N_blobs}')

In [ ]:
# generate particles in the simulation
P = pbm.Particles.from_specs(
    num_particles=(N_EtBr_free, N_blobs),
    D=(D_free, D_bound),
    box=box, rs=rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, 
    t_max=t_max, 
    particles=P, 
    box=box, 
    psf=psf
    )
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
# once simulation is done, it is better to close the HDF5 file to prevent issues in the workflow
# (the file will be opened again in the notebook 2 to generate photon timestamps)
S.store.close()